# C2-linear-models — Review

Work through this notebook *after* the two lesson sessions and (ideally)
the practice sets.
It is a consolidation tool: a summary table, the formula/idiom sheet, a
self-quiz, and pointers to what to redo.
Quiz answers are collapsed at the very end — commit to your answers
before looking.

In [ ]:
import numpy as np

## Concept summary

| Concept | One-line summary | Key fact to retain |
|---|---|---|
| Linear regression | Predict a number as a weighted feature sum plus a bias: $\hat y_i = \sum_k X_{ik} w_k + b$ | one dot product per row; $w_k = 0$ means feature $k$ is ignored; predictions are blends of feature columns $+\ b\,\mathbf 1$ |
| MSE loss | Average squared miss: $\mathrm{MSE} = \frac1n\sum_i(\hat y_i - y_i)^2$ | $= \lVert r\rVert^2/n$ with $r = \hat y - y$; gradient $\partial_{w_j} = \frac{2}{n}\sum_i r_i X_{ij}$, $\partial_b = \frac{2}{n}\sum_i r_i$; residual $\perp$ every column at the optimum |
| L2 regularization | Charge squared weight size: $L = \mathrm{MSE} + \lambda\sum_k w_k^2$ (never $b$) | adds $2\lambda w_j$ to the gradient — a *fading* pull; 1-D closed form $w^\* = a/(1+\lambda)$: shrinks, never exactly zero |
| L1 regularization | Charge absolute weight size: $L = \mathrm{MSE} + \lambda\sum_k \lvert w_k\rvert$ (never $b$) | slope $\pm\lambda$ constant up to the kink; 1-D argmin = soft threshold: $w^\* = 0$ iff $\lvert a\rvert \le \lambda/2$, else $a - \mathrm{sign}(a)\lambda/2$ |
| Sparsity | Many weights *exactly* zero = dropped features | test with `w == 0`, never with rounded printouts; L1 paths hit zero and stay, L2 paths only approach; sparsity = automatic feature selection |

## Formula and idiom sheet

**By hand (the forms the exam's registers expect):**

- Model: $\hat y_i = \sum_k X_{ik} w_k + b$; residual
  $r_i = \hat y_i - y_i$ (this course's pinned convention).
- Loss: $\mathrm{MSE} = \frac1n \sum_i r_i^2$; report plain (held-out)
  MSE for quality — the penalized total is a training objective only.
- MSE gradient: $\partial_{w_j} = \frac{2}{n}\sum_i r_i X_{ij}$,
  $\;\partial_b = \frac{2}{n}\sum_i r_i$.
  (F4 wrote $r = y - \hat y$ with a leading minus — same math; never mix
  the conventions.)
- Ridge: $L = \mathrm{MSE} + \lambda\sum_k w_k^2$; gradient adds
  $2\lambda w_j$; bias unpenalized; 1-D $w^\* = a/(1+\lambda)$.
- Lasso: $L = \mathrm{MSE} + \lambda\sum_k |w_k|$; slopes of
  $\lambda|w|$ are $\pm\lambda$; one-sided slopes at the kink decide;
  1-D soft threshold — zero iff $|a| \le \lambda/2$, else move $a$
  toward $0$ by $\lambda/2$.
- Optimality reading: gradient zero $\iff$ residual orthogonal to every
  feature column and to $\mathbf 1$ ($\sum_i r_i = 0$).

**NumPy idioms (banned-API-safe):**

```python
pred = (X * w).sum(axis=1) + b                    # predictions      (n,)
r = pred - y                                      # residuals        (n,)
mse = np.mean(r**2)                               # mean, not sum
grad_w = (2 / n) * (r[:, None] * X).sum(axis=0)   # (d,) — no np.dot,
grad_b = (2 / n) * r.sum()                        #        no .T, no loops
ridge = mse + lam * np.sum(w**2)                  # penalty: sum over w only
lasso = mse + lam * np.abs(w).sum()
w_star = np.where(np.abs(a) <= lam / 2, 0.0,      # soft threshold,
                  a - np.sign(a) * lam / 2)       # coordinate-wise
zeros = int((w == 0).sum())                       # exact zeros, not "small"
preds_path = (X[None, :, :] * W[:, None, :]).sum(axis=2) + b   # (rows, n)
```

Central-difference check (the checker MAY loop): perturb ONE knob by
$\pm h$ ($h = 10^{-6}$), compare $\big(L(+h) - L(-h)\big)/2h$ against
your partial.
And the standing habit: predict every shape before running — $d$ knobs
need a shape-`(d,)` gradient.

## Self-quiz

Fourteen items, all five concepts covered.
Work by hand (calculator-free), then check against the collapsed answers
at the very end.

1. $w = (2, -1)$, $b = 4$, feature row $x = (3, 5)$: compute $\hat y$.
2. `X` has shape `(80, 4)`. Give the shapes of `w`, `b`, $\hat y$, and
   the MSE gradient with respect to `w`.
3. $\hat y = (3, 0, 2)$, $y = (1, 1, 2)$: compute the residual vector
   and the MSE.
4. Write both MSE gradient formulas (weights and bias) in this course's
   residual convention, and state that convention.
5. At given weights the residuals are $r = (2, -1)$ with
   $X = \begin{pmatrix}1 & 0\\ 3 & 1\end{pmatrix}$, $n = 2$: evaluate
   $\partial\,\mathrm{MSE}/\partial w_0$ and
   $\partial\,\mathrm{MSE}/\partial b$.
6. Why is a gradient that is exactly $-1$ times correct the signature of
   a *convention* bug rather than an arithmetic slip?
7. Write the ridge loss for $\lambda = 0.5$, $w = (2, -2)$, arbitrary
   $b$, if the plain MSE at those weights is $1.3$ — and say why $b$
   does not appear in the penalty.
8. What term does the L2 penalty add to
   $\partial L/\partial w_j$, and what happens to that term as
   $w_j \to 0$?
9. 1-D ridge: minimize $(w - 6)^2 + 2w^2$. Compute $w^\*$.
10. 1-D lasso, $f(w) = (w - a)^2 + \lambda|w|$: compute $w^\*$ for
    $(a, \lambda) = (4, 2)$ and for $(a, \lambda) = (-1, 3)$.
11. State the exact condition on $a$ and $\lambda$ under which the
    lasso's 1-D minimum sits at the kink, and name the two one-sided
    slopes that prove it.
12. In one sentence each: why L1 can finish the push to exactly zero,
    and why L2 cannot.
13. A supplied weight table prints as `[0.01, -0.00, 2.10, 0.00]` at two
    decimals; the exact stored values are
    `[0.011, -0.004, 2.099, 0.0]`. How many exact zeros — and which
    NumPy expression counts them?
14. Write the one-line banned-API-safe NumPy expression for the MSE
    `w`-gradient given `X (n, d)`, `y (n,)`, `w (d,)`, `b` (you may name
    `r` first).

## What to redo, per weak spot

| If you struggled with… | Redo (practice) | Reread (lesson) |
|---|---|---|
| items 1–3 (model, residuals, MSE) | p01, p02, p05, p09 | Session 1 §1–3 |
| items 4–6 (gradients at given weights, conventions) | p04, p06, p13 | Session 1 §5–7 |
| items 7–8 (penalized losses, bias rule) | p07, p08, p16 | Session 2 §2–3, §8 |
| items 9–11 (1-D closed forms, the kink) | p11, p12, p17 | Session 2 §4 |
| item 12 (zeroes-vs-shrinks reasoning) | p03, p11, p15 | Session 2 §3–5 |
| items 13–14 (sparsity counting, banned-API idioms) | p10, p14, p17 | Session 2 §5–7 |
| integration under pressure | p13, p14, p17, p18 | — |

## Self-quiz answers

<details><summary><b>Click to reveal (commit to your answers first)</b></summary>

1. $2\cdot 3 + (-1)\cdot 5 + 4 = 5$.
2. `w`: `(4,)`; `b`: scalar; $\hat y$: `(80,)`; gradient: `(4,)`.
3. $r = (2, -1, 0)$; $\mathrm{MSE} = (4 + 1 + 0)/3 = 5/3$.
4. $r_i = \hat y_i - y_i$ (predictions minus targets);
   $\partial_{w_j} = \frac{2}{n}\sum_i r_i X_{ij}$;
   $\partial_b = \frac{2}{n}\sum_i r_i$.
5. $\partial_{w_0} = \frac{2}{2}(2\cdot 1 + (-1)\cdot 3) = -1$;
   $\partial_b = \frac{2}{2}(2 - 1) = 1$.
6. Because negation is exactly what happens when the residual convention
   and the gradient formula disagree about where the minus lives —
   arithmetic slips rarely negate *every* entry exactly.
7. $L = 1.3 + 0.5\,(4 + 4) = 5.3$; the bias only sets the output level,
   so it is excluded from every penalty in this course.
8. $2\lambda w_j$; it fades to zero along with $w_j$ — the pull dies
   near the origin.
9. $g'(w) = 2(w - 6) + 4w = 6w - 12 = 0 \Rightarrow w^\* = 2$ —
   agreeing with the closed form $a/(1+\lambda) = 6/(1+2) = 2$.
10. $(4, 2)$: $4 > 1 \Rightarrow w^\* = 4 - 1 = 3$.
    $(-1, 3)$: $|-1| \le 1.5 \Rightarrow w^\* = 0$.
11. Kink case iff $|a| \le \lambda/2$; proved by
    $f'(0^+) = \lambda - 2a \ge 0$ and $f'(0^-) = -\lambda - 2a \le 0$.
12. L1's pull has constant size $\lambda$ all the way to the origin, so
    it can beat a weak data slope $2|a|$ there; L2's pull $2\lambda w$
    fades to nothing exactly where the finish line is.
13. One exact zero (the stored `0.0`); `int((w == 0).sum())`.
14. `r = (X * w).sum(axis=1) + b - y` then
    `grad_w = (2 / X.shape[0]) * (r[:, None] * X).sum(axis=0)`.

</details>

---

Scored yourself below ~10/14? Use the redo table above, then retake the
quiz.
At 11+ you are ready for `C3-gradient-descent`, which finally *moves*
the weights you have been measuring.